# Lesson 09 · Vanishing gradients & initialization: measuring the signal through a 20-layer MNIST network

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/09-vanishing-gradient-en.ipynb)

This notebook accompanies [Lesson 09 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/09-vanishing-gradient.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
Why were deep networks once so hard to train? We rebuild exactly the demo's network, a 784 → 20 layers × 64 → 10 MLP
with the very same weights, pass 50 real MNIST images through it, then measure how much of the signal and of the
gradient is left after each layer.

**What you'll do:**
1. Multiply a factor 30 times, then check the formula $\prod_l w_l\, f'(z_l)$ on a chain of 30 neurons.
2. Reproduce the demo's weights bit-exactly (the mulberry32 generator + Box–Muller) and measure the gradient at each
   layer of the 20-layer network.
3. Derive the per-layer factor $\sigma\sqrt{n}\cdot\operatorname{rms}(f')$ yourself, then find the initialization scale
   that keeps it at 1: Xavier, He.
4. Train for real, for 1 epoch: watch the first layer stand still, He beat Xavier with ReLU, and gradient clipping
   stop an explosion.
5. Read the demo's full sweep: 3 activation functions × 4 initialization scales × 4 depths × with/without gradient
   clipping × 5 values of η.

**Background:** [Lesson 02 · Activation functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-kich-hoat/)
(sigmoid, tanh, ReLU, $\sigma'(0) = 0.25$, saturation) and
[Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/) (the backward pass
in matrix form, reused in section 4). The learning rate $\eta$ and how to choose it on a validation set come from
[Lesson 07](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/).
**Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. Only the results of
the full sweep (480 training runs) are loaded from the very file the demo uses, because rerunning it takes a few
minutes; the notebook reruns a few of its configurations and checks that they give the same numbers.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `fmt()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign. This lesson meets very small and
very large numbers, so `num()` prints them as the demo does: ordinary decimals from 0.001 to 10,000, and outside that
range the form $8.5 \cdot 10^{-13}$.

In [ ]:
import base64
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"  # never prints “−0”
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 8.5·10⁻¹³."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{fmt(float(m), digits)}·10{str(int(e)).translate(SUP)}"


def num(x):
    """As in the demo: ordinary decimals from 0.001 to 10,000 (0.0042 · 0.231 · 17.45 · 1,074), else the 10ᵏ form."""
    a = abs(x)
    if 1e-3 <= a < 1e4:
        return fmt(x, 4 if a < 0.01 else 3 if a < 1 else 2 if a < 100 else 0)
    return sci(x)


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure in the English format; call right before plt.show().
    A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(fmt(13007, 0), fmt(0.1, 1), pct(0.9239), num(0.2316), num(17.449), num(8.53e-13), num(2.06e14))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images of 28×28 grayscale handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. As in the demo and in Lesson 07, we split it three ways: the first 55,000 images for training, the last 5,000
images of the training file to *choose* $\eta$ (validation), and the 10,000 test images only to report the final
result.

The cell also loads `mnist-deep.json`, the results file the demo uses (`fetch_json`, with an MD5 check). It holds the
50 images used to measure the signal, and the results of the 480 training runs that we will read in section 8.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels.astype(np.int64)


# The results of a long sweep (the same file as the demo on the web). build.py replaces COURSE_DATA_URL with the data/
# folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, y_all = load("train")
test_img, yte = load("t10k")
DEEP = fetch_json("mnist-deep.json", "b60ee8e1b1e12d7d6f76037f8b662db2")  # the demo's sweep results
X_all = train_img / 255.0
Xtr, ytr = X_all[:55000], y_all[:55000]  # training
Xval, yval = X_all[55000:], y_all[55000:]  # validation: η is chosen here
Xte = test_img / 255.0  # test: for reporting only
print(f"training {fmt(len(ytr), 0)} · validation {fmt(len(yval), 0)} · test {fmt(len(yte), 0)} images")
assert (len(ytr), len(yval), len(yte)) == (DEEP["nTrain"], DEEP["nVal"], DEEP["nTest"])

The demo measures the signal on a small fixed batch: **the first 5 test images of each digit**, 50 images in all. The
cell below picks exactly those images again and checks that they match the images in the demo's file byte for byte.

In [ ]:
pick = np.concatenate([np.flatnonzero(yte == d)[:5] for d in range(10)])
xb, yb = Xte[pick], yte[pick]  # the 50-image batch, x of size 50 × 784
demo_imgs = np.frombuffer(base64.b64decode(DEEP["images"]), np.uint8).reshape(-1, 784)
assert list(pick) == DEEP["mnist"] and (demo_imgs == test_img[pick]).all(), "differs from the demo's 50-image batch"
print(f"batch of {xb.shape[0]} images: indices {pick[:5].tolist()}… match the demo")

fig, axes = plt.subplots(5, 10, figsize=(8, 4.2))
for k, ax in enumerate(axes.T.flat):  # one column per digit
    ax.imshow(xb[k].reshape(28, 28), cmap="gray_r")
    ax.axis("off")
for d in range(10):
    axes[0, d].set_title(str(d), fontsize=11)
plt.show()

Those are the batch's 50 images, one column per digit. Every signal measurement in sections 4–5 and 7 runs on exactly
this batch, as in the demo.

## 3 · Multiplying by one factor thirty times

Imagine a game of telephone through 30 people: each person passes the message on to the next a little more quietly
than they heard it. By the last person, how much of the message is left? Backpropagation (Lesson 06) works exactly
like this: the gradient travels from the last layer back to the first, and at each layer it gets multiplied by a
factor. Lesson 02 computed $0.25^L$, the upper bound on the product of the $\sigma'$. The cell below shows that a
factor doesn't need to be as small as 0.25 to cause trouble: being a little off from 1 is enough.

In [ ]:
for k in [0.25, 0.5, 0.9, 0.99, 1.0, 1.01, 1.1, 2.0]:
    print(f"{fmt(k, 2):>4} multiplied 30 times = {num(k**30)}")
assert f"{0.25**30:.2e}" == "8.67e-19" and f"{0.9**30:.3f}" == "0.042" and f"{1.1**30:.2f}" == "17.45"

A factor of 0.9 sounds harmless, but over 30 layers only 4% of the signal is left; 1.1 swells it 17-fold. Only exactly
1 keeps it unchanged.

What is that factor? Take the simplest network: a chain of 30 neurons, one neuron per layer, $h_l = f(z_l)$ with
$z_l = w_l h_{l-1}$ (no bias). By the chain rule, $\partial h_l / \partial h_{l-1} = f'(z_l)\, w_l$, so

$$\frac{\partial h_{30}}{\partial h_1} = \prod_{l=2}^{30} w_l\, f'(z_l)$$

The factor at each layer is $w_l f'(z_l)$, not just $f'(z_l)$. The cell below defines the lesson's three activation
functions (the same formulas as the demo), computes that product for a tanh chain with $w_l \sim \mathcal N(0, 1)$,
then checks it by nudging $h_1$ (finite differences, as in Lesson 06).

In [ ]:
def act(name, z):
    """f(z) and f′(z) of sigmoid, tanh and ReLU, written exactly as in the demo and in reel/mnist_mlp/deep.py."""
    if name == "sigmoid":
        with np.errstate(over="ignore"):  # very negative z: e^−z overflows to ∞, and 1/(1 + ∞) = 0 is still right
            h = 1 / (1 + np.exp(-z))
        return h, h * (1 - h)
    if name == "tanh":
        h = np.tanh(z)
        return h, 1 - h * h
    if name == "relu":  # convention: f′(0) = 0
        return np.maximum(0, z), (z > 0).astype(float)
    raise ValueError(name)


def chain(f, w, h1):
    """A chain of one neuron per layer: h_l = f(w_l·h_{l−1}). Returns h_1…h_30 and the 29 factors w_l·f′(z_l)."""
    hs, factors = [h1], []
    for wl in w:
        h, d = act(f, wl * hs[-1])
        hs.append(h)
        factors.append(wl * d)
    return np.array(hs), np.array(factors)


eps_chain = np.random.default_rng(0).normal(0, 1, 29)  # w_2 … w_30
_, factors = chain("tanh", eps_chain, 0.5)
nudge = 1e-6
fd = (chain("tanh", eps_chain, 0.5 + nudge)[0][-1] - chain("tanh", eps_chain, 0.5 - nudge)[0][-1]) / (2 * nudge)
print(f"tanh, w ~ N(0, 1): product of the 29 factors w·f′ = {sci(np.prod(factors), 4)};  nudging h₁: {sci(fd, 4)}")
geo = np.exp(np.log(np.abs(factors)).mean())  # geometric mean: the "typical" factor per layer
print(f"|factor| from {fmt(np.abs(factors).min(), 4)} to {fmt(np.abs(factors).max(), 2)}; geometric mean {fmt(geo, 3)}")
assert abs(fd / np.prod(factors) - 1) < 1e-6

The two methods give the same number to the fifth digit: the product formula is right. Some factors are larger than
1, some are close to 0; what decides is their **geometric mean**, because
$|\prod_l a_l| = (\text{geometric mean})^{29}$. Here it is well below 1, so the product shrinks to around $10^{-10}$.

The figure below plots $|\partial h_k / \partial h_1|$ against $k$ for three chains, on a log vertical axis: tanh
(blue circles), sigmoid with the same $w$ (orange squares), and sigmoid with $w$ 4 times larger (green triangles).
The dashed gray line is $0.25^{k-1}$, Lesson 02's bound on the product of the $\sigma'$ alone.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
k = np.arange(1, 31)
for f, scale, color, marker, label in [
    ("tanh", 1, BLUE, "o", "tanh, w ~ N(0, 1)"),
    ("sigmoid", 1, ORANGE, "s", "sigmoid, same w"),
    ("sigmoid", 4, GREEN, "^", "sigmoid, 4× larger w"),
]:
    run_prod = np.abs(np.r_[1, np.cumprod(chain(f, scale * eps_chain, 0.5)[1])])  # |∂h_k/∂h_1|, k = 1…30
    ax.plot(k, run_prod, color=color, marker=marker, ms=4, lw=1.5, label=label)
    print(f"{label:>20}: |∂h₃₀/∂h₁| = {sci(run_prod[-1])}")
ax.plot(k, 0.25 ** (k - 1), color=GREY, ls="--", lw=1.5, label="0.25^(k−1): the σ′ product alone")
print(f"{'0.25^29':>20}: {sci(0.25**29)}")
ax.set_yscale("log")
ax.set(xlabel="layer k", ylabel="|∂h_k / ∂h₁|", title="A chain of 30 neurons: the product of the factors w·f′")
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()

Tanh with $w \sim \mathcal N(0, 1)$ shrinks steadily, even though $\tanh'$ can be as large as 1. Sigmoid with the
same $w$ shrinks even faster than $0.25^{k-1}$, because many $|w_l| < 1$. But sigmoid with $w$ 4 times larger stays
*above* the gray line: large weights make up for the 0.25 limit of $\sigma'$. The real factor is $w \cdot f'$; both
halves matter.

## 4 · The demo's network: 784 → 20 layers × 64 → 10

A real network has a matrix at each layer instead of a number $w_l$. The demo uses a “plain” MLP (no normalization,
no skip connections): $h_l = f(h_{l-1}W_l + b_l)$, $L$ hidden layers of width 64, then 10 logits and a softmax. So
that our results match the demo **bit for bit**, we generate the weights exactly as the browser does:
- the random number generator **mulberry32** with seed 1: each call hashes a 32-bit integer into a uniform number in
  $[0, 1)$;
- the **Box–Muller** transform: each pair $(u_1, u_2)$ becomes one number
  $\varepsilon = \sqrt{-2\ln(1 - u_1)}\,\cos(2\pi u_2)$ with the normal distribution $\mathcal N(0, 1)$;
- the $\varepsilon$ fill, in order, $\varepsilon_{\text{out}}$ (64 × 10), $\varepsilon_1$ (784 × 64), then
  $\varepsilon_2 \dots \varepsilon_{30}$ (64 × 64), row by row.

Then $W_l = \sigma_l\, \varepsilon_l$ and $b_l = 0$. Changing the initialization only changes $\sigma_l$, so every
configuration uses the **same** $\varepsilon$; the 10-layer network is the first 10 layers of the 30-layer network.

In [ ]:
def mulberry32(seed, n):
    """The first n 32-bit integers of mulberry32(seed), the generator the demo (JavaScript) uses, vectorized.
    Call k hashes seed + k·0x6D2B79F5 (mod 2³²). Every operation is on 32-bit integers, so it matches JS bit for bit."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)  # uint32 multiplication wraps around mod 2³², like Math.imul
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return t ^ (t >> 14)


def draw_eps(seed=1, width=64, max_depth=30):
    """ε ~ N(0, 1) for every layer, in exactly the demo's order: ε_out (width × 10), ε₁ (784 × width),
    ε₂…ε_max (width × width)."""
    shapes = [(width, 10), (784, width)] + [(width, width)] * (max_depth - 1)
    sizes = [r * c for r, c in shapes]
    u = mulberry32(seed, 2 * sum(sizes)) / 2**32  # uniform numbers in [0, 1)
    z = np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller: one N(0, 1) number per pair
    eps = [p.reshape(s) for p, s in zip(np.split(z, np.cumsum(sizes)[:-1]), shapes)]
    return eps[1:], eps[0]  # (ε₁ … ε_max), ε_out


EPS = draw_eps(DEEP["seed"], DEEP["width"], DEEP["maxDepth"])
n_eps = sum(e.size for e in EPS[0]) + EPS[1].size
print(f"{fmt(n_eps, 0)} N(0, 1) numbers; the first four integers of mulberry32(1): {mulberry32(1, 4).tolist()}")
print(
    f"ε_out[0, 0] = {fmt(EPS[1][0, 0], 16)};  mean {fmt(np.r_[EPS[1].ravel(), *[e.ravel() for e in EPS[0]]].mean(), 4)}"
)
# The values printed by running the demo's own JavaScript (DeepSignalDemo.tsx) with node
assert mulberry32(1, 4).tolist() == [2693262067, 11749833, 2265367787, 4213581821], "differs from the demo's generator"
assert abs(EPS[1][0, 0] - 1.4043387784152637) < 1e-12 and n_eps == 169600

The first four integers and the first normal number match the demo's JavaScript. Now we define the demo's four
choices of $\sigma$ (section 5 explains each one), the forward pass, and Lesson 06's backward pass. The backward pass
records $\lVert\partial\mathcal L/\partial h_l\rVert$ at each hidden layer: how much of the gradient is left when it
reaches that layer.

In [ ]:
def sigma_of(init, n_in, n_out):
    """The standard deviation of W, as in the demo: small 0.01 · Xavier √(2/(n_in + n_out)) · He √(2/n_in) · large 1.
    init can also be a number g: σ = g/√n_in (section 5)."""
    if not isinstance(init, str):
        return init / np.sqrt(n_in)
    return {"small": 0.01, "xavier": np.sqrt(2 / (n_in + n_out)), "he": np.sqrt(2 / n_in), "large": 1.0}[init]


def init_net(init, depth, eps=EPS):
    eps_h, eps_out = eps
    width = eps_out.shape[0]
    sizes = [784] + [width] * depth
    Ws = [sigma_of(init, sizes[l], width) * eps_h[l] for l in range(depth)] + [sigma_of(init, width, 10) * eps_out]
    return Ws, [np.zeros(W.shape[1]) for W in Ws]


def forward(f, Ws, bs, x):
    """hs: x, then h₁ … h_L; ds: f′(z) of each hidden layer; also returns the 10 logits."""
    hs, ds = [x], []
    for W, b in zip(Ws[:-1], bs[:-1]):
        h, d = act(f, hs[-1] @ W + b)
        hs.append(h)
        ds.append(d)
    return hs, ds, hs[-1] @ Ws[-1] + bs[-1]


def ce_loss(z, y):
    """The mean cross-entropy, computed with log-sum-exp, so that very large logits still give a finite loss."""
    zmax = z.max(1)
    return float(np.mean(zmax + np.log(np.exp(z - zmax[:, None]).sum(1)) - z[np.arange(len(y)), y]))


def backward(Ws, hs, ds, z, y):
    """Backpropagation (Lesson 06) for softmax + cross-entropy: the gradients of every W and b, and ‖∂L/∂h_l‖ of
    each hidden layer."""
    e = np.exp(z - z.max(1, keepdims=True))
    dz = e / e.sum(1, keepdims=True)
    dz[np.arange(len(y)), y] -= 1  # ∂L/∂z = p − onehot(y), divided by the batch size
    dz /= len(y)
    gW, gb, gh = [hs[-1].T @ dz], [dz.sum(0)], []
    dh = dz @ Ws[-1].T
    for l in reversed(range(len(ds))):
        gh.insert(0, np.linalg.norm(dh))
        da = dh * ds[l]  # through the activation function: multiply by f′(z)
        gW.insert(0, hs[l].T @ da)
        gb.insert(0, da.sum(0))
        dh = da @ Ws[l].T  # through the weights: multiply by Wᵀ
    return gW, gb, np.array(gh)

The cell below wraps all of the demo's measurements into one function, `signal()`, which runs on the 50-image batch
with an *untrained* network:
- **forward pass:** how different the 50 images still are at layer $l$ (the standard deviation across images of
  each neuron, RMS over the 64 neurons), and the fraction of neurons that are saturated (sigmoid: $h < 0.03$ or
  $h > 0.97$; tanh: $|h| > 0.97$) or dead (ReLU: 0 for all 50 images);
- **backward pass:** $\lVert\partial\mathcal L/\partial h_l\rVert$ at each layer, the norm of the whole gradient, and
  the **per-layer factor**: the geometric mean
  $(\lVert\partial\mathcal L/\partial h_1\rVert / \lVert\partial\mathcal L/\partial h_L\rVert)^{1/(L-1)}$.

It also computes a predicted number, `pred`, which section 5 will explain.

In [ ]:
def signal(f, init, depth, x=xb, y=yb, eps=EPS):
    """All of the demo's measurements on an untrained network (like compute() in DeepSignalDemo.tsx)."""
    Ws, bs = init_net(init, depth, eps)
    hs, ds, z = forward(f, Ws, bs, x)
    gW, gb, gh = backward(Ws, hs, ds, z, y)
    n_in = [W.shape[0] for W in Ws[:-1]]
    sig_w = [sigma_of(init, n, Ws[0].shape[1]) for n in n_in]
    pred = np.array([s * np.sqrt(n) * np.sqrt((d**2).mean()) for s, n, d in zip(sig_w, n_in, ds)])  # section 5
    if f == "relu":
        sat = np.array([(h <= 0).all(0).mean() for h in hs[1:]])  # dead neurons: 0 for the whole batch
    else:
        lim = (0.03, 0.97) if f == "sigmoid" else (-0.97, 0.97)
        sat = np.array([((h < lim[0]) | (h > lim[1])).mean() for h in hs[1:]])
    return {
        "spread": np.array([np.sqrt(h.var(0).mean()) for h in hs[1:]]),  # how different the images still are
        "gh": gh,
        "pred": pred,
        "sat": sat,
        "loss": ce_loss(z, y),
        "gnorm": float(np.sqrt(sum((g**2).sum() for g in gW + gb))),
        "factor": (gh[0] / gh[-1]) ** (1 / (depth - 1)),
        "pred_factor": float(np.exp(np.log(pred[1:]).mean())),
        "ratio": gh[0] / gh[-1],
    }


FLOOR = 1e-15  # as in the demo: a standard deviation below this is only rounding error, the images are all alike


def report(f, init, depth):
    s = signal(f, init, depth)
    same = np.flatnonzero(s["spread"] < FLOOR)
    print(
        f"{f}/{init}, {depth} layers: each layer multiplies the gradient by {num(s['factor'])} "
        f"(predicted {num(s['pred_factor'])}); layer 1 relative to layer {depth}: {num(s['ratio'])}\n"
        f"   difference between images: layer 1 {num(s['spread'][0])}, layer {depth} {num(s['spread'][-1])}"
        + (f" (= 0 from layer {same[0] + 1})" if len(same) else "")
        + f"; {'dead neurons' if f == 'relu' else 'saturated'} {pct(s['sat'].mean(), 0)}; loss {num(s['loss'])}"
    )
    return s

### 🤔 Predict first
The demo's default configuration: sigmoid, Xavier initialization, 20 layers. Lesson 02 says each $\sigma'$ is at
most 0.25. So is the gradient at layer 1, relative to layer 20, about $0.25^{19} \approx 3.6 \cdot 10^{-12}$, larger
than that, or smaller? And in the forward pass, are the 50 different images still different at layer 20?

In [ ]:
s_sig = report("sigmoid", "xavier", 20)
print(f"0.25^19 = {sci(0.25**19)}")
# deep.py prints the same ratios when it makes the demo's file (live_stats, the same 50 images, the same weights)
assert abs(s_sig["ratio"] / 8.53346e-13 - 1) < 1e-5, "differs from the demo"
assert (
    f"{s_sig['factor']:.2f}" == "0.23" and f"{s_sig['pred_factor']:.2f}" == "0.24" and f"{s_sig['loss']:.2f}" == "2.50"
)
fwd = (s_sig["spread"][-1] / s_sig["spread"][0]) ** (1 / 19)
print(
    f"forward pass: the difference between images shrinks ×{fmt(fwd, 3)} per layer (backward pass: ×{fmt(s_sig['factor'], 3)})"
)
r10 = signal("sigmoid", "xavier", 10)["ratio"]
print(f"same configuration, 10 layers: layer 1 relative to layer 10 = {num(r10)}, i.e. {fmt(1 / r10, 0)} times smaller")

Smaller even than $0.25^{19}$: at layer 1, the gradient is only $8.5 \cdot 10^{-13}$ times the gradient at layer 20.
Going back, each layer multiplies it by 0.232, less than 0.25 because not every neuron sits exactly at $z = 0$, where
$\sigma'$ is largest. Even with 10 layers, the gradient at the first layer is already hundreds of thousands of times
smaller than at the last.

The forward pass is even worse. At layer 1, the images differ by about 0.081; at layer 20, only by
$8.3 \cdot 10^{-14}$: all 50 images give almost the same vector. That is no surprise: the forward and backward passes
go through the **same** Jacobian matrices, so the *difference* between two inputs shrinks by nearly the same factor:
×0.234 per layer in the forward pass, ×0.232 in the backward pass. The network has forgotten the input image before
it gets a chance to learn. And not a single neuron is saturated (0%): the gradient vanishes here without any need for
saturation.

The figure below plots both passes by layer, like the demo's two charts, with a log vertical axis. The blue line
(round dots) is sigmoid + Xavier; the dashed orange line (square dots) is the configuration the demo pins for
comparison, **ReLU + He**. The dotted gray line is “magnitude kept”: the level of layer 1 in the forward pass, the
level of layer 20 in the backward pass.

In [ ]:
s_relu = signal("relu", "he", 20)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
layers = np.arange(1, 21)
for ax, key, title in [
    (axes[0], "spread", "Forward pass: how different are the 50 images?"),
    (axes[1], "gh", "Backward pass: ‖∂L/∂hₗ‖"),
]:
    ax.plot(layers, s_sig[key], color=BLUE, marker="o", ms=4, lw=2, label="sigmoid · Xavier")
    ax.plot(layers, s_relu[key], color=ORANGE, marker="s", ms=4, lw=2, ls="--", label="ReLU · He")
    ref = s_sig[key][0] if key == "spread" else s_sig[key][-1]
    ax.axhline(ref, color=GREY, ls=":", lw=1.5, label="magnitude kept")
    ax.set_yscale("log")
    ax.set(title=title, xlabel="layer l", xticks=[1, 5, 10, 15, 20])
    ax.grid(alpha=0.3)
axes[0].set_ylabel("standard deviation across images")
axes[1].text(20, 1e-12, "← the backward pass goes from layer 20 to layer 1", ha="right", fontsize=9)
axes[0].legend(frameon=False, fontsize=9, loc="lower left")
plt.tight_layout()
fmt_axes(fig)
plt.show()
print(
    f"ReLU · He: layer 1 relative to layer 20 = {num(s_relu['ratio'])}; each layer multiplies by {num(s_relu['factor'])}"
)
assert abs(s_relu["ratio"] / 0.889264 - 1) < 1e-5, "differs from the demo"

With ReLU + He, the gradient goes through 20 layers almost unchanged: layer 1 receives 0.889 times the gradient of
layer 20, that is, each layer multiplies it by 0.994. The sigmoid line drops straight down, in both charts. The next
part asks: what decides the per-layer factor, and why does He give exactly 1?

## 5 · The per-layer factor: $\sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f')$

The backward pass through layer $l$ (Lesson 06) is $\partial\mathcal L/\partial h_{l-1} = \big(\partial\mathcal L/\partial h_l \odot f'(z_l)\big)\,W_l^\top$.
Assume the entries of $W_l$ are independent, with mean 0 and standard deviation $\sigma$, and that $W_l$ has
$n_{\text{in}}$ rows. Each entry of the resulting vector is a sum of many random terms, so the squared norm of that
vector is approximately

$$\Big\lVert \frac{\partial\mathcal L}{\partial h_{l-1}} \Big\rVert^2 \approx n_{\text{in}}\,\sigma^2\,\operatorname{rms}(f')^2\,\Big\lVert \frac{\partial\mathcal L}{\partial h_l} \Big\rVert^2
\quad\Longrightarrow\quad \text{factor per layer} \approx \sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f'(z_l))$$

where $\operatorname{rms}(f')$ is the square root of the mean of $f'^2$ over every neuron and every image. That is
exactly `pred` in `signal()`. To make the factor equal to 1, we choose $\sigma$ according to the activation function:
- **tanh** has $f' \approx 1$ near 0, so it needs $\sigma = 1/\sqrt{n_{\text{in}}}$. Glorot & Bengio (2010) ask for
  both the forward pass ($n_{\text{in}}\operatorname{Var}(w) = 1$) and the backward pass
  ($n_{\text{out}}\operatorname{Var}(w) = 1$), then pick a compromise: **Xavier**,
  $\operatorname{Var}(w) = 2/(n_{\text{in}} + n_{\text{out}})$. At a 64 → 64 layer, the three conditions coincide.
  (The paper draws $w$ from a uniform distribution with the same variance; the demo uses a normal distribution.)
- **ReLU** has $f' = 1$ for about half of the neurons and 0 for the other half, so
  $\operatorname{rms}(f') \approx \sqrt{1/2}$ and it needs $\sigma = \sqrt{2/n_{\text{in}}}$: **He** (He et al., 2015),
  $\operatorname{Var}(w) = 2/n_{\text{in}}$.
- **sigmoid** has $f' \le 0.25$, so it needs $\sigma\sqrt{n_{\text{in}}} \ge 4$. But a large $\sigma$ makes $z$ large,
  and a large $z$ makes $f'$ smaller. This section measures where those two forces meet.

The cell below prints $\sigma$ for the demo's four initializations at a 64 → 64 layer.

In [ ]:
INITS = {"small": "small σ = 0.01", "xavier": "Xavier", "he": "He", "large": "large σ = 1"}
for init, label in INITS.items():
    s64 = sigma_of(init, 64, 64)
    print(f"{label:>14}: σ = {fmt(s64, 3)},  σ·√64 = {fmt(s64 * 8, 3)}")
assert f"{sigma_of('xavier', 64, 64):.3f}" == "0.125" and f"{sigma_of('he', 64, 64):.3f}" == "0.177"  # as in the demo

### 🤔 Predict first
With tanh and $\sigma = 1$ (“large”), $z$ is very large, so most neurons saturate: $|h| > 0.97$, where
$\tanh'(z) < 0.06$. Will the gradient through 20 layers **vanish** or **explode**?

In [ ]:
s_tl = report("tanh", "large", 20)
rms_fp = s_tl["pred"][1:] / 8  # pred = σ·√n·rms(f′) with σ = 1, n = 64
print(f"rms(f′) at layers 2–20: from {fmt(rms_fp.min(), 3)} to {fmt(rms_fp.max(), 3)};  σ·√n = 8")
assert f"{s_tl['factor']:.2f}" == "2.08" and f"{s_tl['sat'].mean():.2f}" == "0.78"

Explode: ×2.08 per layer, so layer 1 receives more than a million times the gradient of layer 20, even though 78% of
the neurons are saturated. “Saturated means a zero gradient” is a misconception. $\operatorname{rms}(f')$ only drops
to 0.248–0.279, because the few unsaturated neurons still contribute $f' \approx 1$. Multiplied by
$\sigma\sqrt{n} = 8$, the factor is still larger than 2. Conversely, sigmoid + Xavier in section 4 has 0% saturated
neurons, and yet its gradient vanishes. Saturation and vanishing gradients are two different things. What decides is
the product $\sigma\sqrt{n}\cdot\operatorname{rms}(f')$.

The cell below measures the per-layer factor for all 12 of the demo's configurations (3 functions × 4
initializations, 20 layers), next to the predicted number.

In [ ]:
TABLE = {(f, init): signal(f, init, 20) for f in ["sigmoid", "tanh", "relu"] for init in INITS}
print("per-layer factor, 20 layers: measured (predicted σ√n·rms(f′))\n")
print(f"{'':>8}" + "".join(f"{label:>17}" for label in INITS.values()))
for f in ["sigmoid", "tanh", "relu"]:
    cells = [f"{num(TABLE[f, i]['factor'])} ({num(TABLE[f, i]['pred_factor'])})" for i in INITS]
    print(f"{f:>8}" + "".join(f"{c:>17}" for c in cells))
print(f"\nReLU · Xavier: layer 1 relative to layer 20 = {num(TABLE['relu', 'xavier']['ratio'])}")
print(f"sigmoid · σ = 1: {pct(TABLE['sigmoid', 'large']['sat'].mean(), 0)} of the neurons saturated")
worst = max(abs(s["factor"] / s["pred_factor"] - 1) for s in TABLE.values())
print(f"measured and predicted differ by at most {pct(worst, 1)}")
# deep.py (live_stats) prints these ratios for the demo
assert abs(TABLE["tanh", "xavier"]["ratio"] / 0.309564 - 1) < 1e-5
assert abs(TABLE["relu", "xavier"]["ratio"] / 0.00119368 - 1) < 1e-5

The one-line formula predicts the measured factor in all 12 configurations, to within a few percent. The table's
three rows teach three lessons:
- **sigmoid** vanishes with every one of the demo's initializations, He included. Only $\sigma = 1$
  ($\sigma\sqrt n = 8$) lifts the factor to 0.883, and the price is 50% of the neurons saturated.
- **tanh** needs Xavier (0.940) or He (1.07): both are close to 1, one shrinking slightly, the other swelling
  slightly.
- **ReLU** with Xavier shrinks ×0.702 per layer, so the gradient at layer 1 is only 0.0012 times that of layer 20.
  The reason is exactly one missing factor of $\sqrt 2$: Xavier gives $\sigma\sqrt n = 1$, but ReLU switches off about
  half of the neurons, so $\operatorname{rms}(f') \approx 0.71$. He adds exactly that factor and gives 0.994.

The figure below sweeps the initialization scale $g = \sigma\sqrt{n_{\text{in}}}$ continuously (used for every layer,
the input and output layers included), then finds by bisection the value of $g$ that makes the factor exactly 1 for
each function. Both axes are logarithmic. The gray lines mark the $g$ of Xavier and He at a 64 → 64 layer.

In [ ]:
gains = np.geomspace(0.25, 16, 25)
STYLE = {"sigmoid": (ORANGE, "s", "--"), "tanh": (BLUE, "o", "-"), "relu": (GREEN, "^", ":")}
fig, ax = plt.subplots(figsize=(8, 3.8))
crit = {}
for f, (color, marker, ls) in STYLE.items():
    label = "ReLU" if f == "relu" else f
    ax.plot(gains, [signal(f, g, 20)["factor"] for g in gains], color=color, marker=marker, ms=4, ls=ls, label=label)
    lo, hi = 0.25, 16.0  # bisection on a log scale: the factor increases with g around where it crosses 1
    for _ in range(40):
        mid = math.sqrt(lo * hi)
        lo, hi = (mid, hi) if signal(f, mid, 20)["factor"] < 1 else (lo, mid)
    crit[f] = lo
    s = signal(f, lo, 20)
    print(
        f"{f:>7}: factor = 1 at g = {fmt(lo, 3)};  there {pct(s['sat'].mean(), 0)} of the neurons are "
        f"{'dead' if f == 'relu' else 'saturated'}, rms(f′) = {fmt(s['pred_factor'] / lo, 3)}"
    )
for g, name in [(1, "Xavier"), (math.sqrt(2), "He")]:
    ax.axvline(g, color=GREY, lw=1)
    ax.text(g * 1.03, 6, name, fontsize=9, color="#555555")
ax.axhline(1, color="black", lw=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(
    xlabel="initialization scale g = σ·√n_in",
    ylabel="factor per layer",
    title="The per-layer factor against the initialization scale (20 layers)",
)
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()
homog = signal("relu", 2.0, 20)["factor"] / signal("relu", 1.0, 20)["factor"]
print(
    f"ReLU: doubling g multiplies the factor by {fmt(homog, 3)};  sigmoid needs σ = g/√64 = {fmt(crit['sigmoid'] / 8, 2)}"
)
assert abs(homog - 2) < 0.05 and abs(crit["relu"] - math.sqrt(2)) < 0.02 and 9 < crit["sigmoid"] < 11

- **ReLU** crosses the line 1 at $g = 1.423$, right next to He's $\sqrt 2 \approx 1.414$. Its curve is a perfect
  straight line on the log-log scale: doubling $g$ multiplies the factor by 2.019. Since
  $\operatorname{ReLU}(cz) = c\operatorname{ReLU}(z)$ for $c > 0$, scaling every weight doesn't change which neurons
  are on, so each layer of the backward pass multiplies by exactly one more $c$; the deviation from 2 comes from the
  error signal $p - \text{onehot}(y)$ at the output, because the softmax of larger logits changes its direction.
- **tanh** crosses at $g = 1.124$, between Xavier and He. Beyond that, the curve gradually flattens, because large
  $z$ starts to saturate tanh.
- **sigmoid** needs $g = 10.07$, that is, $\sigma \approx 1.26$ at a 64 → 64 layer, and by then more than half of
  the neurons are already saturated. No initialization scale keeps both the gradient and the neurons in their active
  region. Besides, sigmoid is always positive, so every later layer receives inputs with a nonzero mean. Glorot &
  Bengio (2010) point out exactly this: sigmoid is “unsuited for deep networks with random initialization”, and it
  drives the top hidden layer into saturation.

### He keeps the magnitude, not the differences

In section 4's figure, the orange ReLU + He line is flat in the backward pass, but in the forward pass it still goes
slowly down. Two different quantities are at play here: the **magnitude** of the signal (the RMS of $h_l$) and the
**difference between images** (the standard deviation across images, the left chart). The cell below measures both,
plus how similar two images are: the mean cosine between the $z_l$ vectors of the 50 images (1 means the same
direction).

In [ ]:
def similarity(f, init, depth):
    Ws, bs = init_net(init, depth)
    hs, _, _ = forward(f, Ws, bs, xb)
    zs = [h @ W for h, W in zip(hs[:-1], Ws[:-1])]  # z_l = h_{l−1}·W_l (b = 0)
    unit = [z / np.linalg.norm(z, axis=1, keepdims=True) for z in zs]
    cos = [(u @ u.T)[np.triu_indices(len(u), 1)].mean() for u in unit]  # mean over the 1,225 pairs of images
    return np.sqrt([(h**2).mean() for h in hs[1:]]), np.sqrt([h.var(0).mean() for h in hs[1:]]), np.array(cos)


for f, init in [("relu", "he"), ("tanh", "xavier")]:
    rms, spread, cos = similarity(f, init, 20)
    print(
        f"{f}/{init}: magnitude (RMS) {fmt(rms[0], 3)} → {fmt(rms[-1], 3)};  difference between images {fmt(spread[0], 3)} → "
        f"{fmt(spread[-1], 3)};  cosine between two images {fmt(cos[0], 3)} → {fmt(cos[-1], 3)}  (layer 1 → layer 20)"
    )
    assert f == "relu" or cos[-1] < cos[0], "with tanh + Xavier, the cosine does not rise"

With ReLU + He, the **magnitude** of the signal is almost preserved across 20 layers (the RMS goes from 0.288 down to
0.186, fluctuating around that level; a finite network is never perfectly flat). But any two images become more and
more **alike**: the cosine between them rises from 0.317 to 0.963. Each ReLU cuts off the negative part, so the
vectors gradually crowd into the same direction. With tanh + Xavier, the cosine does not rise; it even falls, from
0.317 to 0.238: tanh is symmetric around 0 and has no such tendency. Initializing at the right scale preserves the
*magnitude*, but not necessarily the *information*. Schoenholz et al. (2017) study exactly this phenomenon: in a
random network, information can only propagate through a limited depth, unless the network sits close to the
boundary between “shrinking” and “swelling”.

## 6 · Real training: early layers stand still, late layers keep learning

So far we have only measured an *untrained* network. Does a small gradient at the first layers really spoil
learning? The cell below trains exactly as `reel/mnist_mlp/deep.py` does for the demo: 1 epoch on 55,000 images,
batches of 64 images in the same shuffled order (seed 0), plain SGD, with or without **gradient clipping**
(section 7). After training, it measures the accuracy on the validation set and on the test set, and how far each
layer has moved from where it started: $\lVert W_l - W_l^{(0)}\rVert / \lVert W_l^{(0)}\rVert$.

In [ ]:
BATCH, CLIP = DEEP["batch"], DEEP["clip"]  # 64 images per batch; clip the global norm of the gradient at 1
ORDER = np.random.default_rng(0).permutation(len(ytr))  # the same batch order as deep.py


def accuracy(f, Ws, bs, X, y):
    return float((forward(f, Ws, bs, X)[2].argmax(1) == y).mean())


def rel_change(W, W0):
    """‖W − W₀‖ / ‖W₀‖, computed so that it doesn't overflow even when the weights have swelled to 10²⁰⁰."""
    d = W - W0
    m, m0 = np.abs(d).max(), np.abs(W0).max()
    return float(m * np.linalg.norm(d / m) / (m0 * np.linalg.norm(W0 / m0))) if m > 0 else 0.0


def train(f, init, depth, eta, clip=False, eps=EPS):
    """1 epoch of SGD as in deep.py. Returns the accuracies, how far each layer moved, and the parameters; or the step
    at which the loss became NaN."""
    Ws, bs = init_net(init, depth, eps)
    W0 = [W.copy() for W in Ws]
    with np.errstate(over="ignore", invalid="ignore"):  # exploding configurations overflow; we detect it via the loss
        for step, i in enumerate(range(0, len(ORDER), BATCH), 1):
            idx = ORDER[i : i + BATCH]
            hs, ds, z = forward(f, Ws, bs, Xtr[idx])
            if not np.isfinite(ce_loss(z, ytr[idx])):
                return {"nan_step": step}
            gW, gb, _ = backward(Ws, hs, ds, z, ytr[idx])
            scale = 1.0
            if clip:  # clipping by the global norm: divide every gradient by the same number if ‖g‖ > 1
                norm = np.sqrt(sum((g**2).sum() for g in gW + gb))
                scale = min(1.0, CLIP / norm) if np.isfinite(norm) else 0.0
            for l in range(depth + 1):
                Ws[l] -= eta * scale * gW[l]
                bs[l] -= eta * scale * gb[l]
        if not all(np.isfinite(W).all() for W in Ws):
            return {"nan_step": step}
        return {
            "val": accuracy(f, Ws, bs, Xval, yval),
            "test": accuracy(f, Ws, bs, Xte, yte),
            "move": [rel_change(W, W_) for W, W_ in zip(Ws, W0)],
            "params": (Ws, bs),
        }


def demo_run(f, init, depth, clip=False):
    """The matching configuration in the demo's file (η chosen on the validation set, as in deep.py)."""
    return next(c for c in DEEP["configs"] if (c["act"], c["init"], c["depth"], c["clip"]) == (f, init, depth, clip))

### 🤔 Predict first
Train the 20-layer sigmoid + Xavier network with $\eta = 0.01$, the value the demo chose on the validation set. After
one epoch (860 steps), what percentage of the test images does the network get right? And how far do the weights of
layer 1 move, relative to their initial magnitude?

In [ ]:
r_sig = train("sigmoid", "xavier", 20, eta=0.01)
pred_sig = forward("sigmoid", *r_sig["params"], Xte)[2].argmax(1)
print(f"{-(-len(ORDER) // BATCH)} steps; test accuracy {pct(r_sig['test'])}")
print(
    f"the network predicts the digits {np.unique(pred_sig).tolist()} for all {fmt(len(pred_sig), 0)} images; 1s make up {pct((yte == 1).mean())}"
)
print(
    f"layer 1 moved {num(r_sig['move'][0])} of its initial magnitude, layer 20 moved {num(r_sig['move'][19])}, the output layer {num(r_sig['move'][-1])}"
)
d = demo_run("sigmoid", "xavier", 20)
print(f"demo: η = {fmt(d['eta'], 2)}, test {pct(d['test'])}, layer 1 moved {num(d['move'][0])}")
assert abs(r_sig["test"] - d["test"]) < 1e-9 and (pred_sig == 1).all(), "differs from the demo"
assert abs(r_sig["move"][0] / d["move"][0] - 1) < 0.01 and abs(r_sig["move"][-1] / d["move"][-1] - 1) < 0.01
print(
    f"the digit 1 in the training set: {fmt((ytr == 1).sum(), 0)}/{fmt(len(ytr), 0)} images, the most of the 10 digits"
)
assert np.bincount(ytr).argmax() == 1

11.35%, exactly the share of 1s in the test set: the network answers “1” for **every** image. The weights of layer 1
moved by $2.1 \cdot 10^{-13}$ times their initial magnitude, so they stood still; the output layer moved 0.107, so it
is still learning. But what did the output layer learn? Section 4 showed that at layer 20, all 50 images already give
almost the same vector, so the only thing the output layer can learn is **which digit is most frequent**. The digit 1
is the most common digit in the training set. “Late layers keep learning” is true, but they learn from an input that
has lost all information about the image.

### He against Xavier for ReLU, three seeds

Now for the test of He et al. (2015): the same 20-layer ReLU network, initialized with He or with Xavier. Each uses
the $\eta$ the demo chose on the validation set (He: 0.03; Xavier: 0.1). To know whether the difference exceeds random
variation, we run 3 seeds of the weight generator: seed 1 is the demo's network, seeds 2 and 3 give different
$\varepsilon$. The batch order stays the same.

In [ ]:
runs = {}
for init in ["he", "xavier"]:
    eta = demo_run("relu", init, 20)["eta"]
    for seed in [1, 2, 3]:
        eps = EPS if seed == DEEP["seed"] else draw_eps(seed)
        runs[init, seed] = train("relu", init, 20, eta, eps=eps)
    accs = [runs[init, s]["test"] for s in [1, 2, 3]]
    print(
        f"ReLU · {INITS[init]:<6} (η = {fmt(eta, 2)}): {' · '.join(pct(a) for a in accs)}   range {pct(min(accs))}–{pct(max(accs))}"
    )
he = [runs["he", s]["test"] for s in [1, 2, 3]]
xa = [runs["xavier", s]["test"] for s in [1, 2, 3]]
print(
    f"demo (seed 1): He {pct(demo_run('relu', 'he', 20)['test'])}, Xavier {pct(demo_run('relu', 'xavier', 20)['test'])}"
)
print(
    "He against Xavier:",
    "beyond the variation between seeds" if min(he) > max(xa) else "within the variation between seeds",
)
print(
    f"per-layer factor of ReLU · Xavier at the start: {num(TABLE['relu', 'xavier']['factor'])}, to the power 19 = {num(TABLE['relu', 'xavier']['factor'] ** 19)}"
)
# Training this deep network is sensitive to rounding error (section 9), so on a different CPU, seed 1 can be off from
# the demo by a few tenths of a point: we only check it with a wide tolerance, plus the main conclusion.
assert abs(he[0] - demo_run("relu", "he", 20)["test"]) < 0.03 and min(he) > max(xa) + 0.1

He beats Xavier far beyond the variation between seeds: the three He networks reach 89.84–92.39%, the three Xavier
networks only 11.35–50.18%, and one Xavier network learns nothing at all (11.35%, “always guess 1” again). That is a
miniature version of Figure 3 in the paper by He et al.: with a 30-layer network, Xavier initialization “completely
stalls”, while their initialization converges. The difference is only a factor of $\sqrt 2$ in the standard
deviation of the 64 → 64 layers (0.177 versus 0.125), but over 19 layers it becomes $0.702^{19} = 0.0012$.

For the three runs we just made (seed 1, which is also the demo's network), the figure below plots how far each layer
has moved: $\lVert W_l - W_l^{(0)}\rVert / \lVert W_l^{(0)}\rVert$ against layer $l$, with layer 21 the output
layer. The vertical axis is logarithmic.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for r, color, marker, ls, label in [
    (r_sig, BLUE, "o", "-", "sigmoid · Xavier"),
    (runs["xavier", 1], GREEN, "^", ":", "ReLU · Xavier"),
    (runs["he", 1], ORANGE, "s", "--", "ReLU · He"),
]:
    ax.plot(np.arange(1, 22), r["move"], color=color, marker=marker, ms=4, ls=ls, lw=1.8, label=label)
    print(
        f"{label:>16}: layer 1 moved {num(r['move'][0])}, layer 20 moved {num(r['move'][19])}, output layer {num(r['move'][20])}"
    )
ax.set_yscale("log")
ax.set(
    xlabel="layer (21 = output layer)", ylabel="‖W − W₀‖ / ‖W₀‖", title="After 1 epoch: how far has each layer moved?"
)
ax.set_xticks([1, 5, 10, 15, 20, 21])
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()

With sigmoid, the blue line climbs from $10^{-13}$ at layer 1 to around $10^{-1}$ at the output layer: each layer
closer to the output learns far more, exactly the picture of “early layers stand still, late layers keep learning”.
With ReLU, both initializations make every layer move by a few percent to a few tens of percent, layer 1 included. In
ReLU + Xavier, the first layer doesn't stand still: the ratio 0.0012 only holds at initialization, and it learns with
a larger $\eta = 0.1$. But moving a lot doesn't mean learning well: this network only reaches 37.09%.

## 7 · Exploding gradients and gradient clipping

The flip side: ReLU with $\sigma = 1$. Each layer multiplies the gradient by 5.67 (the table in section 5).

### 🤔 Predict first
A 20-layer ReLU network, $\sigma = 1$, untrained. Roughly how large is the initial cross-entropy loss on the 50
images? (A network that spreads its guesses evenly over the 10 digits has a loss of $\ln 10 \approx 2.30$.) And what
happens when we train it with $\eta = 0.03$?

In [ ]:
s_big = report("relu", "large", 20)
print(f"norm of the whole gradient: {num(s_big['gnorm'])};  ln 10 = {fmt(math.log(10), 2)}")
r_big = train("relu", "large", 20, eta=0.03)
print(f"without gradient clipping, η = 0.03: the loss becomes NaN at step {r_big['nan_step']}")
d_big = demo_run("relu", "large", 20)
print("demo, every η:", " · ".join(f"η = {fmt(s['eta'], 2)}: NaN at step {s['nanStep']}" for s in d_big["sweep"]))
assert r_big["nan_step"] == d_big["sweep"][1]["nanStep"] == 3, "differs from the demo"

The initial loss is $7.4 \cdot 10^{15}$ instead of 2.30: the logits are gigantic, and the network is absolutely
certain of the wrong answers. The norm of the gradient is $2.0 \cdot 10^{16}$, so the very first SGD step, however
small $\eta$ is within the demo's grid, throws the weights extremely far. By step 3 (step 2 with $\eta = 1$), the
numbers overflow the range of 64-bit floating-point numbers and the loss becomes NaN (“not a number”).

**Gradient clipping** by the global norm: if the norm $\lVert g\rVert$ of the *whole* gradient (every $W$ and every
$b$ together) exceeds a threshold $c$, divide all of it by the same number so that the norm becomes exactly $c$:

$$g \leftarrow g \cdot \min\!\Big(1, \frac{c}{\lVert g\rVert}\Big)$$

Pascanu et al. (2013) proposed it for RNNs, where exploding gradients are an everyday occurrence. It is still standard
when training large language models: GPT-3 and LLaMA both “clip the global norm of the gradient at 1.0”. The demo
uses $c = 1$. The cell below trains the same network again, this time with clipping.

In [ ]:
r_clip = train("relu", "large", 20, eta=0.03, clip=True)
d_clip = demo_run("relu", "large", 20, clip=True)
print(
    f"with gradient clipping, η = 0.03: test accuracy {pct(r_clip['test'])} (demo: {pct(d_clip['test'])}, η = {fmt(d_clip['eta'], 2)})"
)
print(
    f"layer 1 moved {num(r_clip['move'][0])} of its initial magnitude, the output layer moved {num(r_clip['move'][-1])}"
)
W_norm = math.sqrt(sum((W**2).sum() for W in init_net("large", 20)[0]))
print(f"each step moves at most η·c = {fmt(0.03 * CLIP, 2)};  initial ‖W‖ of the whole network = {fmt(W_norm, 0)}")
assert abs(r_clip["test"] - d_clip["test"]) < 0.01, "differs from the demo"

With clipping, the network no longer blows up: each step moves at most $\eta c = 0.03$, tiny compared with the
weights' magnitude of 359. But it barely learns either: 16.26%, and layer 1 moves only 0.0021 of its initial
magnitude. Gradient clipping blocks steps that are too large; it doesn't fix a network in which every layer
multiplies the signal by 5.67.

Nor does it fix vanishing gradients, for two reasons that the cell below prints:
- clipping divides **every** gradient by the same number, so the ratio between layer 1 and the last layer doesn't
  change;
- when the gradient vanishes, the global norm is usually already below the threshold, so clipping never happens.

In [ ]:
scale = CLIP / s_big["gnorm"]
print(
    f"ReLU · σ = 1: clipping multiplies every gradient by {sci(scale)}; layer 1 relative to layer 20 before clipping {num(s_big['ratio'])}, "
    f"after clipping {num((scale * s_big['gh'][0]) / (scale * s_big['gh'][-1]))}"
)
print(f"sigmoid · Xavier: ‖g‖ = {num(s_sig['gnorm'])} < {fmt(CLIP, 0)}, so there is nothing to clip")
d0, d1 = demo_run("sigmoid", "xavier", 20), demo_run("sigmoid", "xavier", 20, clip=True)
print(f"demo, sigmoid · Xavier · 20 layers: without clipping {pct(d0['test'])}, with clipping {pct(d1['test'])}")
assert s_sig["gnorm"] < CLIP  # below the threshold: clipping never happens

## 8 · The demo's whole experiment

The demo runs exactly the `train()` function above for every combination: 3 activation functions × 4 initializations
× 4 depths (5, 10, 20, 30 layers) × with/without gradient clipping × 5 values of $\eta$ (0.01 · 0.03 · 0.1 · 0.3 ·
1), that is, 480 training runs, which take a few minutes. For each configuration, $\eta$ is chosen by the accuracy
on the validation set, then measured once on the test set. We read the results from `DEEP["configs"]`. A run that
diverged is recorded as `{"eta": …, "nanStep": step}` instead of `"val"`; if every $\eta$ diverges, `eta`, `val`,
`test` and `move` are `null` (`None` in Python). The file contains no NaN or infinity.

In [ ]:
DEPTHS, ACTS = DEEP["depths"], ["sigmoid", "tanh", "relu"]
n_runs = sum(len(c["sweep"]) for c in DEEP["configs"])
n_nan = sum("nanStep" in s for c in DEEP["configs"] for s in c["sweep"])
print(
    f"{len(DEEP['configs'])} configurations × {len(DEEP['etas'])} values of η = {n_runs} training runs; the loss became NaN in {n_nan} of them"
)


def cell(c):
    return "NaN" if c["test"] is None else fmt(100 * c["test"], 1)


print(
    f"\ntest accuracy (%), without / with clipping; NaN = every η diverges\n{'':>22}"
    + "".join(f"{d:>11} layers" for d in DEPTHS)
)
for f in ACTS:
    for init, label in INITS.items():
        row = [f"{cell(demo_run(f, init, d))} / {cell(demo_run(f, init, d, True))}" for d in DEPTHS]
        print(f"{f:>7} {label:>14}" + "".join(f"{r:>18}" for r in row))

The figure below plots the same table (without gradient clipping) against depth, one panel per activation function.
Each initialization has its own color, marker shape and line style. The “NaN” points (every $\eta$ diverges) are
drawn as × marks at the bottom of the panel. The dotted gray horizontal line is 11.35%, the level of “always guess
the digit 1”.

In [ ]:
INIT_STYLE = {
    "small": (GREY, "x", (0, (1, 2))),
    "xavier": (BLUE, "o", "-"),
    "he": (ORANGE, "s", "--"),
    "large": (GREEN, "^", "-."),
}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
for ax, f in zip(axes, ACTS):
    for init, (color, marker, ls) in INIT_STYLE.items():
        acc = [demo_run(f, init, d)["test"] for d in DEPTHS]
        ok = [a is not None for a in acc]
        ax.plot(
            [d for d, k in zip(DEPTHS, ok) if k],
            [100 * a for a, k in zip(acc, ok) if k],
            color=color, marker=marker, ls=ls, lw=1.8, ms=6, label=INITS[init],
        )  # fmt: skip
        if not all(ok):
            ax.scatter([d for d, k in zip(DEPTHS, ok) if not k], [2] * ok.count(False), color=color, marker="X", s=60)
    ax.axhline(100 * (yte == 1).mean(), color=GREY, ls=":", lw=1)
    ax.set(title="ReLU" if f == "relu" else f, xlabel="number of hidden layers", xticks=DEPTHS, ylim=(0, 100))
    ax.grid(alpha=0.3)
axes[2].text(30, 6, "NaN", ha="center", fontsize=9)
axes[0].set_ylabel("test accuracy (%)")
axes[0].legend(frameon=False, fontsize=8, loc="center left")
plt.tight_layout()
fmt_axes(fig)
plt.show()
for f, init in [("sigmoid", "large"), ("relu", "xavier"), ("relu", "he")]:
    print(f"{f} · {INITS[init]}: " + " · ".join(f"{d} layers {cell(demo_run(f, init, d))}%" for d in DEPTHS))
small = [c["test"] for c in DEEP["configs"] if c["init"] == "small"]
print(f"σ = 0.01, all {len(small)} cells: from {pct(min(small), 1)} to {pct(max(small), 1)}")
tanh_ok = [c["test"] for c in DEEP["configs"] if c["act"] == "tanh" and c["init"] in ("xavier", "he")]
print(f"tanh · Xavier or He, all {len(tanh_ok)} cells: lowest {pct(min(tanh_ok), 1)}")
print(f"ReLU · σ = 1 · 10 layers: with clipping {cell(demo_run('relu', 'large', 10, True))}%")

Reading the table and the figure (each cell is one seed, with $\eta$ chosen on the validation set):
- **σ = 0.01** learns nothing at any depth, activation function or $\eta$: 11.3% in all 24 cells, that is, “always
  guess 1” (its lines overlap at the bottom of each panel, together with the sigmoid + Xavier line).
  Each layer multiplies the gradient by about 0.02–0.08, so even 5 layers are enough for the first layer to stand
  still.
- **sigmoid** learns only with σ = 1, and gets worse with depth: 86.2% at 5 layers, 27.0% at 30 layers: a factor of
  0.883 per layer is still below 1, and half of the neurons are saturated.
- **tanh** with Xavier or He learns well at every depth: 91.3% at the lowest.
- **ReLU** with Xavier is fine at 5–10 layers (94.8% and 94.1%) and then collapses at 20–30 layers (37.1% and
  20.0%): a shallow network doesn't “notice” the factor of $\sqrt 2$, a deep one does. He et al. saw exactly this
  too: their 22-layer network converges with both methods, their 30-layer network doesn't with Xavier. He holds 92.4%
  at 20 layers, but also drops to 80.4% at 30 layers (section 10 discusses why).
- **σ = 1 with ReLU** diverges (NaN) for every $\eta$ from 10 layers up. Gradient clipping turns NaN into a network
  that runs (26.4% at 10 layers), but not into a good network.

Each cell has only one seed. In section 6, three seeds of the 20-layer ReLU + He network were already a few points
apart, so differences of that size between with and without gradient clipping may just be variation between seeds.

## 9 · Try changing things

The cells below rerun `signal()` and `train()` with other parameters. You can change the values and run them again.
`signal()` is only one forward pass and one backward pass on 50 images; each `train()` call is a full epoch, so it is
much slower.

### The width of the network

The formula $\sigma\sqrt{n}\cdot\operatorname{rms}(f')$ is an average: it is more accurate when each layer has many
neurons. The cell below measures the per-layer factor of ReLU + He, 20 layers, at widths 16, 64 and 256, with 3
seeds of the weight generator for each width.

In [ ]:
by_width = {}
for width in [16, 64, 256]:
    fs = by_width[width] = [signal("relu", "he", 20, eps=draw_eps(seed, width, 20))["factor"] for seed in [1, 2, 3]]
    print(
        f"width {width:>3}: per-layer factor {' · '.join(fmt(v, 3) for v in fs)}  (range {fmt(min(fs), 3)}–{fmt(max(fs), 3)})"
    )
f_low = min(by_width[16])
print(f"width 16, lowest seed: {fmt(f_low, 3)} to the power 19 = {num(f_low**19)}")

On average they are all around 1, but the narrow 16-neuron network varies much more from seed to seed; over 19
layers, a factor of 0.892 has already become 0.114. This is why “Xavier” and “He” are rules about **variance**: they
only hold on average, and they hold better the wider the network.

### Depth

The per-layer factor hardly changes with depth; what changes is how many times it gets multiplied. The cell below
prints the ratio between layer 1 and the last layer at 5, 10, 20 and 30 layers.

In [ ]:
for f, init in [("sigmoid", "xavier"), ("relu", "xavier"), ("relu", "he"), ("tanh", "large")]:
    cells = [num(signal(f, init, d)["ratio"]) for d in DEPTHS]
    print(f"{f}/{init:<6}: " + " · ".join(f"{d} layers {c}" for d, c in zip(DEPTHS, cells)))

Sigmoid + Xavier loses about 6 orders of magnitude every 10 layers; tanh with σ = 1 gains about 3. ReLU + He drifts
slowly (0.980 at 10 layers, 0.829 at 30 layers): a factor of 0.994 is not exactly 1, and 29 multiplications still
leave a trace.

### Rounding error

The cell below retrains the 20-layer ReLU + Xavier network (seed 1, $\eta = 0.1$, as in section 6), but multiplies
every $\varepsilon$ by $1 + 10^{-14}\cdot$noise: a change in the 14th decimal place, about the size of
floating-point rounding error.

In [ ]:
rng = np.random.default_rng(7)
nudge_h = [e * (1 + 1e-14 * rng.standard_normal(e.shape)) for e in EPS[0]]  # ε₁ … ε₃₀
eps_nudged = (nudge_h, EPS[1] * (1 + 1e-14 * rng.standard_normal(EPS[1].shape)))  # then ε_out
r_n = train("relu", "xavier", 20, demo_run("relu", "xavier", 20)["eta"], eps=eps_nudged)
print(f"ReLU · Xavier · 20 layers: original {pct(runs['xavier', 1]['test'])}, weights off by 10⁻¹⁴: {pct(r_n['test'])}")

A change in the 14th digit changes the result after one epoch, often by a few percentage points. This deep network
is sensitive: the tiniest differences get amplified through 860 steps and 20 layers. So when you run the notebook on
a different machine (Colab uses an x86 CPU, the demo was made on aarch64), a few of the training runs in section 6
may come out slightly off from the demo's numbers, even with the same code and the same seed. The conclusions don't
change: the `assert` cells only check those runs with a wide tolerance. For a difference to be believed, it must be
larger than the range of variation between seeds, like He against Xavier in section 6.

## 10 · Limits and next steps

Initializing at the right scale only takes care of **the start**. The cell below measures the per-layer factor of the
20-layer ReLU + He network from section 6 again, this time *after* 1 epoch of training, on the same 50 images.

In [ ]:
Ws_he, bs_he = runs["he", 1]["params"]
hs, ds, z = forward("relu", Ws_he, bs_he, xb)
_, _, gh_after = backward(Ws_he, hs, ds, z, yb)
f_after = (gh_after[0] / gh_after[-1]) ** (1 / 19)
rms_after = [np.sqrt((h**2).mean()) for h in hs[1:]]
print(
    f"ReLU · He, 20 layers: per-layer factor at the start {fmt(s_relu['factor'], 3)}, after 1 epoch {fmt(f_after, 3)}"
)
print(
    f"magnitude (RMS) of h: layer 1 {fmt(rms_after[0], 3)}, layer 20 {fmt(rms_after[-1], 3)} (at the start 0.288 and 0.186)"
)

After one epoch, the factor has moved away from 1, and the magnitude of the signal swells about 4-fold from layer 1
to layer 20: training changes the weights, so the distribution of $z$ at each layer drifts, and nothing pulls it
back. At 30 layers, ReLU + He already drops to 80.4% (section 8). He et al. observed this too: their 30-layer model
converges, but makes more errors than the 14-layer model (top-1 error on ImageNet 38.56% versus 33.82%). Lesson 10
and Lesson 13 fix exactly this:
- [Lesson 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chuan-hoa/):
  renormalize $z$ at **every** layer, throughout training, instead of choosing the right scale only once at the
  start;
- [Lesson 13 · Skip connections & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/):
  $h_l = h_{l-1} + F(h_{l-1})$, so the Jacobian of each layer is $I + \partial F/\partial h$ and the gradient always
  has a direct path back to the first layer.

Nor are vanishing gradients only a problem of networks with many layers. An RNN
([Lesson 15](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn/)) uses *the same* matrix at every time step, so
over 100 steps the gradient gets multiplied 100 times by the same factor. That is exactly the problem that
Hochreiter (1991) and Bengio, Simard & Frasconi (1994) analyzed, before the name “deep learning” existed.

## Summary
- The gradient at the first layer is a product of many factors $w_l\,f'(z_l)$: 0.9 multiplied 30 times leaves
  0.042; 1.1 multiplied 30 times becomes 17.45.
- On the demo's 20-layer network, sigmoid + Xavier multiplies the gradient by 0.232 per layer: layer 1 receives only
  $8.5 \cdot 10^{-13}$ times the gradient of layer 20, and the 50 images become the same vector, even though no
  neuron is saturated.
- The per-layer factor $\approx \sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f')$. Xavier chooses $\operatorname{Var}(w) = 2/(n_{\text{in}} + n_{\text{out}})$
  for tanh (0.940), He chooses $2/n_{\text{in}}$ for ReLU (0.994); ReLU with Xavier is missing exactly $\sqrt 2$
  (0.702).
- Saturation is not the same as vanishing gradients: tanh with σ = 1 has 78% of its neurons saturated, yet its
  gradient still explodes, ×2.08 per layer.
- After 1 epoch, the 20-layer ReLU network reaches 89.84–92.39% with He but 11.35–50.18% with Xavier (3 seeds);
  sigmoid + Xavier guesses “1” for every image, with layer 1 moving $2.1 \cdot 10^{-13}$.
- Gradient clipping (global norm ≤ 1, as in GPT-3 and LLaMA) turns a network that diverges at step 3 into one that
  runs, but it fixes neither vanishing gradients nor a wrong initialization.

## Review questions
1. In a chain with one neuron per layer, what is the gradient's factor at each layer? Why can large weights make up
   for the 0.25 limit of $\sigma'$, and what is the price?
2. The 20-layer sigmoid + Xavier network has 0% saturated neurons, and yet its gradient vanishes. Tanh with σ = 1,
   on the other hand, has 78% saturated neurons, and its gradient explodes. Explain both with the formula
   $\sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f')$.
3. Why does He use $\operatorname{Var}(w) = 2/n_{\text{in}}$ for ReLU, twice tanh's $1/n_{\text{in}}$? If you use
   Xavier for ReLU, how much is lost at each layer, and how much is left after 19 layers?
4. After 1 epoch, the sigmoid + Xavier network gets 11.35% of the test images right. Where does that number come
   from? Which layer learned, and what did it learn?
5. What does clipping by the global norm do to the ratio between the gradient at layer 1 and at the last layer? Why
   does it cure exploding gradients but not vanishing ones?
6. With ReLU + He, the magnitude of the signal is almost preserved across 20 layers. So why is the 30-layer network
   still worse than the 20-layer one?

<details><summary>Hints and answers</summary>

1. $w_l\,f'(z_l)$, not just $f'(z_l)$. With a large $|w|$, the product can exceed 1 even though $f' \le 0.25$; but a
   large $w$ makes $z$ large, pushing the neurons into saturation (σ = 1: 50% of the sigmoid neurons saturated), and
   the network struggles to learn.
2. Sigmoid + Xavier: $\sigma\sqrt n = 1$ and $\operatorname{rms}(\sigma') < 0.25$, so the factor is 0.232 even though
   no neuron is saturated. Tanh with σ = 1: $\sigma\sqrt n = 8$, while $\operatorname{rms}(f') \approx 0.27$ thanks to
   the few neurons that are not yet saturated; the product is 2.08 > 1.
3. ReLU switches off about half of the neurons, so $\operatorname{rms}(f') \approx \sqrt{1/2}$; for the product to be
   1 you need $\sigma\sqrt{n} = \sqrt 2$. With Xavier, each layer multiplies by 0.702, and after 19 layers 0.0012 is
   left.
4. It is the share of 1s in the test set: the network guesses “1” for every image. Only the output layer learned
   (layer 1 moved $2.1 \cdot 10^{-13}$), but its input was already the same for every image, so all it could learn
   was which digit is most frequent.
5. Nothing changes: every gradient is divided by the same number. It blocks steps that are too large, so the network
   no longer blows up into NaN. When the gradient vanishes, the global norm is usually already below the threshold
   (sigmoid + Xavier: 0.830 < 1), so there is nothing to clip.
6. He only holds at initialization, and only preserves the *magnitude*: the images still gradually crowd into the
   same direction (the cosine goes from 0.317 to 0.963 across 20 layers), and after training the distribution of $z$
   drifts. Normalization (Lesson 10) and skip connections (Lesson 13) fix this part.
</details>

## Further reading
- S. Hochreiter (1991), [*Untersuchungen zu dynamischen neuronalen Netzen*](https://people.idsia.ch/~juergen/SeppHochreiter1991ThesisAdvisorSchmidhuber.pdf),
  thesis (in German): the first analysis of vanishing gradients.
- Y. Bengio, P. Simard & P. Frasconi (1994), [Learning long-term dependencies with gradient descent is difficult](https://doi.org/10.1109/72.279181).
- X. Glorot & Y. Bengio (2010), [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html):
  sigmoid and saturation, Xavier initialization (equations 12 and 16).
- K. He et al. (2015), [Delving Deep into Rectifiers](https://arxiv.org/abs/1502.01852): He initialization; Figures 2–3
  compare it with Xavier on 22- and 30-layer networks.
- R. Pascanu, T. Mikolov & Y. Bengio (2013), [On the difficulty of training recurrent neural networks](https://arxiv.org/abs/1211.5063):
  gradient clipping by the norm.
- S. Schoenholz et al. (2017), [Deep Information Propagation](https://arxiv.org/abs/1611.01232): how far information
  propagates through a random network.
- T. Brown et al. (2020), [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) (GPT-3, Appendix B) and
  H. Touvron et al. (2023), [LLaMA](https://arxiv.org/abs/2302.13971) (section 2.3): clipping the gradient norm at 1.0.
- deeplearning.ai, [Initializing neural networks](https://www.deeplearning.ai/ai-notes/initialization/) (interactive);
  [Dive into Deep Learning, section 5.4](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html);
  the CS231n notes, [Neural Networks Part 2](https://cs231n.github.io/neural-networks-2/) (initialization).

**Next lesson:** [Lesson 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chuan-hoa/):
normalize $z$ at every layer throughout training, so that the signal keeps its scale not only at initialization.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")